# Kaggle Inference — Qwen2.5-VL-3B-Instruct (Data Pipeline Incident Response)

Runs the schema-drift agent using **Qwen/Qwen2.5-VL-3B-Instruct** (this is the HuggingFace repo name for the Qwen3-VL 4B model).

**VRAM budget on Kaggle T4 (15 GB):**
- Model weights in 4-bit NF4: ~2.5 GB
- Vision encoder (resident even for text tasks): ~0.8 GB
- KV cache + activations (max_new_tokens=512, history=16 turns): ~1.5 GB
- Framework overhead: ~0.4 GB
- **Total: ~5.2 GB** — leaves 9+ GB headroom, very safe.

---

## Kaggle Secrets required

Add these in **Settings → Secrets** before running:

| Secret name | What it is | Minimum permission |
|---|---|---|
| `GITHUB_TOKEN` | Classic Personal Access Token from github.com/settings/tokens | `repo` scope (needed to clone a private repo). If the repo is public, only `public_repo` is enough. |
| `HF_TOKEN` | HuggingFace token from huggingface.co/settings/tokens | **Read** access. If `Qwen/Qwen2.5-VL-3B-Instruct` is gated, you also need to accept the model license on the HF model page once while logged in — the token itself stays read-only. |

> **HF_TOKEN write access is NOT needed for inference.** Only needed later if you push a fine-tuned model to the Hub (training notebook).


In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient

_s           = UserSecretsClient()
GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN')
HF_TOKEN     = _s.get_secret('HF_TOKEN')

# Expose to transformers / huggingface_hub automatically
os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN   # legacy alias still used by some libs

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git'
REPO_DIR = '/kaggle/working/Meta_hackathon'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull --quiet')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo ready:', os.getcwd())


In [ ]:
# Install deps — qwen-vl-utils handles the Qwen2.5-VL image/video preprocessing pipeline
# We only do text inference here but the package is required by the model class
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'bitsandbytes',
    'qwen-vl-utils', 'pandas', 'numpy', 'openai', 'python-dotenv'], check=True)

import bitsandbytes as bnb, torch
print(f'bitsandbytes : {bnb.__version__}')
print(f'torch        : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Qwen/Qwen2.5-VL-3B-Instruct is the HuggingFace model ID for the Qwen3-VL 4B class model.
# The product is called 'Qwen3-VL'; HF uses the Qwen2.5-VL series naming for the repo.
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# ── 4-bit NF4 quantization ─────────────────────────────────────────────
# T4 does NOT support bfloat16 natively — use float16 for compute dtype.
# double_quant: quantizes the quantization constants too (~0.1% quality, ~0 VRAM cost).
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── Processor (VL models use processor, not just tokenizer) ────────────
# min_pixels / max_pixels caps the resolution the vision encoder processes.
# For text-only inference this prevents accidental large image buffer allocation.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

# ── Model load ──────────────────────────────────────────────────────────
# device_map='auto' places layers on GPU first, spills to CPU if needed.
# With 4-bit the full 3B model is ~2.5 GB on GPU — leaves 12+ GB for KV cache.
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.eval()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded : {MODEL_ID}')
print(f'VRAM alloc   : {allocated:.2f} GB')
print(f'VRAM reserved: {reserved:.2f} GB')
print(f'VRAM free    : {total - reserved:.2f} GB  (of {total:.1f} GB)')


In [ ]:
import re, json, textwrap
from typing import Optional, List
from src.models import PipelineAction, PipelineObservation

MODEL_NAME  = MODEL_ID
MAX_TOKENS  = 512    # JSON actions are short; 512 is generous and keeps KV cache small
TEMPERATURE = 0.1
MAX_STEPS   = 25
BENCHMARK   = 'data_pipeline_incident_response'

FALLBACK_ACTION = PipelineAction(action_type='compare_schema', params={'table': 'raw_orders'})


def _strip_think(text: str) -> str:
    """Remove <think>...</think> blocks.

    Qwen3 models emit chain-of-thought inside <think> tags before their answer.
    These are useful for debugging but break JSON parsing — strip them before
    trying to extract the action object.
    """
    return re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()


def generate(messages: list) -> str:
    """One inference pass through Qwen2.5-VL for text-only chat.

    Important differences vs LLaMA:
      - Must use processor.apply_chat_template, not tokenizer.apply_chat_template
      - Must call processor(text=...) to get input tensors (handles the VL preprocessing)
      - Must slice output_ids by input_len to get only new tokens
      - Must strip <think> tags from output before returning
    """
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], padding=True, return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=max(TEMPERATURE, 0.01),
            do_sample=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    raw = processor.tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True)
    return _strip_think(raw)


def parse_action(text: str) -> PipelineAction:
    """Extract a PipelineAction from model output with repair fallback."""
    if not text:
        return FALLBACK_ACTION
    # Strip markdown fences
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```'))
    start = text.find('{')
    end   = text.rfind('}') + 1
    if start == -1 or end == 0:
        return FALLBACK_ACTION
    # Try clean parse first
    try:
        return PipelineAction(**json.loads(text[start:end]))
    except Exception:
        pass
    # Repair truncated JSON
    for suffix in ['"}}', '}}', '}']:
        try:
            return PipelineAction(**json.loads(text[start:] + suffix))
        except Exception:
            continue
    return FALLBACK_ACTION


print('Helpers ready.')


In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
You are an expert data engineer diagnosing and fixing broken data pipelines.
Choose ONE action each turn. Respond with ONLY a single JSON object. No markdown, no prose.

WORKFLOW:
1. read_data_sample on the failing table first.
2. check_schema / compare_schema if type or rename drift suspected.
3. If compare_schema shows renamed/missing columns, call handle_drift first.
4. Apply fix: add_data_filter or patch_transformation.
5. run_pipeline to verify. Repeat if still failing.
6. alert_upstream_team ONLY when data is genuinely unfixable.

PATCH TYPES: cast_column | coalesce | dedup | parse_currency | strip_prefix
IMPORTANT: After parse_currency, ALWAYS chain coalesce on the same column before run_pipeline.
Exception: if column is a denominator (e.g. impressions in CTR), filter IS NOT NULL instead of coalesce.
If roas_summary has 0 rows, the join key mismatches. compare_schema -> strip_prefix -> cast_column.

HANDLE DRIFT strategies: detect | resolve_column_rename | numeric_format | null_fill |
  type_cast | join_key_prefix | filter_invalid | alert_upstream

RULES:
- ONLY JSON. No explanation.
- Never fix before reading data (-0.5 penalty).
- Never use mark_acceptable (-1.0 penalty).
- Stop when pipeline_passed is true (unless alert still required).
""").strip()


def build_prompt(obs, step: int) -> str:
    failed  = '\n'.join(
        f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}'
        for r in obs.failed_assertions
    ) or '  (none)'
    passed  = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag     = '\n'.join(
        f'  {n.step_id}: {n.input_table} -> {n.output_table}'
        + (f' | filters:{n.applied_filters}' if n.applied_filters else '')
        + (f' | patches:{n.applied_patches}' if n.applied_patches else '')
        for n in obs.dag_structure
    )
    hist    = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs[-2:])
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-5:]) or '  (none)'

    sample = ''
    if obs.data_sample:
        rows      = obs.data_sample[:4]
        null_rows = [r for r in obs.data_sample if any(v is None for v in r.values())]
        sample    = '\nDATA SAMPLE:\n' + json.dumps(rows, default=str)
        if null_rows:
            sample += f'\nNULL ROWS ({len(null_rows)} found):\n' + json.dumps(null_rows[:3], default=str)

    schema = ''
    if obs.current_schema:
        schema += '\nSCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff:
        schema += '\nSCHEMA DIFF: ' + json.dumps(obs.schema_diff)

    reads = sum(1 for a in obs.actions_taken if any(k in a for k in ('read_data', 'check_schema', 'compare')))
    fixes = sum(1 for a in obs.actions_taken if any(k in a for k in ('add_data_filter', 'patch_trans', 'handle_drift')))
    recent_runs = sum(1 for a in obs.actions_taken[-3:] if 'run_pipeline' in a)
    hint = ''
    if reads >= 2 and fixes == 0:
        hint = '\n[HINT] You have read enough data. Apply a fix now.'
    if recent_runs >= 2 and not obs.pipeline_passed:
        hint += '\n[HINT] Do not call run_pipeline again without applying a fix first.'

    return textwrap.dedent(f"""
    STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})
    DESCRIPTION: {obs.description}
    PIPELINE PASSED: {obs.pipeline_passed}
    LAST ACTION RESULT: {obs.last_action_result}
    DAG:
    {dag}
    FAILING:
    {failed}
    PASSING: {passed}
    HISTORY:
    {hist}
    RECENT ACTIONS:
    {actions}
    {sample}{schema}{hint}
    Respond with exactly ONE action JSON object.
    """).strip()


print('Prompts ready.')


In [ ]:
from src.environment import DataPipelineEnv


def log_start(task, env, mdl):
    print(f'[START] task={task} env={env} model={mdl}', flush=True)

def log_step(step, action, reward, done):
    a = str(action).replace('\n', ' ')[:200]
    print(f'[STEP] step={step} action={a} reward={reward:.2f} done={str(done).lower()} error=null', flush=True)

def log_end(success, steps, score, rewards):
    r = ','.join(f'{x:.2f}' for x in rewards)
    print(f'[END] success={str(success).lower()} steps={steps} score={score:.2f} rewards={r}', flush=True)


def run_episode(task_id: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    env     = DataPipelineEnv(task_id=task_id, max_steps=max_steps)
    history: List[dict] = []
    rewards: List[float] = []
    steps_taken = 0
    score = 0.01
    success = False
    n_passed = n_total = 0
    pipeline_passed = False

    log_start(task_id, BENCHMARK, MODEL_NAME)

    try:
        obs = env.reset()
        if verbose:
            print(f'\n{"="*60}', file=sys.stderr)
            print(f'TASK: {task_id.upper()}  |  {len(obs.failed_assertions)} assertions failing', file=sys.stderr)
            print(f'{"="*60}', file=sys.stderr)

        for step in range(1, max_steps + 1):
            if obs.pipeline_passed:
                if verbose:
                    print(f'[PASSED] Pipeline passed at step {step-1}!', file=sys.stderr)
                break

            user_text = build_prompt(obs, step)
            history.append({'role': 'user', 'content': user_text})
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history

            # Clear KV cache fragments before each generation
            # Qwen2.5-VL's vision encoder stays resident and can cause fragmentation
            torch.cuda.empty_cache()

            response_text = ''
            try:
                response_text = generate(messages)
            except torch.cuda.OutOfMemoryError:
                # OOM recovery: aggressively trim history and retry
                print(f'[OOM] Step {step}: trimming history to 8 turns and retrying.', file=sys.stderr)
                history = history[-8:]
                messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history
                torch.cuda.empty_cache()
                try:
                    response_text = generate(messages)
                except Exception as e2:
                    print(f'[OOM-RETRY-FAIL] {e2}', file=sys.stderr)
            except Exception as exc:
                print(f'[ERR] Step {step}: {exc}', file=sys.stderr)

            action = parse_action(response_text)

            # Safe fallback when model returns empty: use diagnostic action
            if not response_text.strip() and obs.failed_assertions:
                action = PipelineAction(
                    action_type='compare_schema',
                    params={'table': obs.failed_assertions[0].table}
                )

            history.append({'role': 'assistant', 'content': response_text or '{}'})
            # Tighter history bound than LLaMA: VL model uses more memory per token
            if len(history) > 16:
                history = history[-16:]

            result = env.step(action)
            obs    = result.observation
            reward = result.reward or 0.0
            done   = result.done
            rewards.append(reward)
            steps_taken = step

            log_step(step, json.dumps(action.model_dump())[:200], reward, done)

            if verbose:
                print(
                    f'[Step {step}] {action.action_type}  reward={reward:+.2f}  '
                    f'passed={len(obs.passed_assertions)}/{len(obs.failed_assertions)+len(obs.passed_assertions)}  '
                    f'| {obs.last_action_result[:80]}',
                    file=sys.stderr
                )

            if done:
                break

        n_total  = len(obs.failed_assertions) + len(obs.passed_assertions)
        n_passed = len(obs.passed_assertions)
        pipeline_passed = obs.pipeline_passed
        score   = min(max(n_passed / n_total if n_total else 0.0, 0.01), 0.99)
        success = score >= 0.1

        if verbose:
            print(f'\n--- Summary ---', file=sys.stderr)
            print(f'  score={score:.2f}  reward={sum(rewards):.2f}  steps={steps_taken}  passed={pipeline_passed}', file=sys.stderr)

    except Exception as exc:
        import traceback
        print(f'[ERROR] {task_id}: {exc}', file=sys.stderr)
        traceback.print_exc(file=sys.stderr)
    finally:
        try: env.close()
        except: pass
        log_end(success, steps_taken, score, rewards)

    return {
        'task_id': task_id, 'score': round(score, 4), 'success': success,
        'pipeline_passed': pipeline_passed, 'total_reward': round(sum(rewards), 4),
        'steps_taken': steps_taken, 'assertions_passed': n_passed, 'assertions_total': n_total,
    }


print('Runner ready.')


In [ ]:
# Change TASKS_TO_RUN to run a single task for faster testing
# Options: 'easy' | 'medium' | 'hard' | 'hard2' | 'all'
TASKS_TO_RUN = 'all'

task_ids = ['easy', 'medium', 'hard', 'hard2'] if TASKS_TO_RUN == 'all' else [TASKS_TO_RUN]

all_results = []
for task_id in task_ids:
    result = run_episode(task_id=task_id, max_steps=MAX_STEPS, verbose=True)
    all_results.append(result)
    torch.cuda.empty_cache()   # release KV cache between tasks

print('\n' + '='*60, file=sys.stderr)
print('FINAL SCORES', file=sys.stderr)
print('='*60, file=sys.stderr)
total = 0.0
for r in all_results:
    status = '[PASSED]' if r['pipeline_passed'] else '[FAILED]'
    print(f"  {r['task_id']:8s} | score={r['score']:.2f} | "
          f"reward={r['total_reward']:+.2f} | steps={r['steps_taken']:2d} | {status}", file=sys.stderr)
    total += r['score']
print(f'\n  Avg score: {total/len(all_results):.4f}', file=sys.stderr)
print('\nJSON_RESULTS:', json.dumps(all_results, indent=2), file=sys.stderr)


In [ ]:
# Run any time to check VRAM health
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total_vr  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM allocated : {allocated:.2f} GB')
print(f'VRAM reserved  : {reserved:.2f} GB')
print(f'VRAM free      : {total_vr - reserved:.2f} GB  (of {total_vr:.1f} GB)')
